In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('merge/result/f2_XGB_파라미터 튜닝.csv')

print(df['Predicted_Segment'].value_counts())

Predicted_Segment
E    496313
D     76735
C     26468
A       459
B        25
Name: count, dtype: int64


In [3]:
df = df.rename(columns={'Predicted_Segment': 'Segment'})

In [4]:
# 각 ID별로 Segment 최빈값 구하기
mode_segment_per_id = (
    df.groupby('ID')['Segment']
    .agg(lambda x: x.mode().iloc[0]) 
    .reset_index()
    .rename(columns={'Segment': 'Segment_mode'})
)

# 원본 데이터와 결합 (각 row에 ID별 최빈 Segment 정보 붙이기)
df_merged = df.merge(mode_segment_per_id, on='ID')

# ID별로 최빈 Segment와 같은 Segment만 필터링
df_filtered = df_merged[df_merged['Segment'] == df_merged['Segment_mode']]

# 그중에서 ID당 1개 row만 남기기
df_final = df_filtered.drop_duplicates(subset='ID', keep='first')

# 불필요한 보조 컬럼 제거 (Segment_mode)
df_final = df_final.drop(columns=['Segment_mode']).reset_index(drop=True)

In [5]:
print(df['Segment'].value_counts())

Segment
E    496313
D     76735
C     26468
A       459
B        25
Name: count, dtype: int64


In [6]:
df_final.to_csv('merge/result/F3_XGB_예측_중복제거(최빈값).csv', index=False, encoding='utf-8-sig')